<a href="https://colab.research.google.com/github/ryanmart25/bird-song-recognizer/blob/utils_setup/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bird Species Identifier
A model for identifying the species of bird from audio.

## Imports

In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path
import os
import sys
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D, MaxPooling2D, concatenate, Conv1D, MaxPooling1D
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_curve, auc, mean_squared_error
import matplotlib.pyplot as plt
from collections.abc import Sequence
from sklearn import preprocessing
%matplotlib inline
import csv
import glob
from IPython.display import Image
import seaborn as sns


## Global Control Flow Flags and Program Configuration

In [11]:
ITERATION = 0
PAUL = True # paul, you are running in an environment with a different keras backend than us, and are using pytorch instead of tensorflow.
# use this flag to gate code that should be run when only you want it to run. If this feels like a clunky and bad idea, feel free to disregard this.
# In general, my idea for these flags was they could be used to section off highly experimental / broken code, or code that only works in a specific
# environment that others might not have.
RYAN = True
BEN = True
WINDOW_SIZE = 7
OPTIMIZER_LEARNING_RATE = 0.001


## Define Helper Methods

In [12]:
def plot_losses(history, base_path, iteration:int):
    # Plot training & validation loss over epochs
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.ylim(bottom=0.0, top=10.0)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs. Validation Loss")
    plt.legend()
    plt.savefig(
        os.path.join(base_path, f"training-validiation-loss--epoch---Model {iteration}")
    )
    plt.close()


def print_schema(dataframe: pd.DataFrame):
    print('~~~~~~dataframe schema~~~~~~')
    print(f"Dataframe shape: {dataframe.shape} | Dataframe length: {len(dataframe)}")
    print('Column labels: ')
    print(dataframe.columns)
    print('Dataframe head: ')
    print(f"{dataframe.head()}")
def print_column(dataframe: pd.DataFrame, columns: str | list[str]):
    if isinstance(columns, list):
        for i, label in enumerate(columns):
            print(f"column {i}")
            print(dataframe[label])
    else:
        print(dataframe[columns])
# Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1] for red,green,blue)
def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Encode text values to indexes(i.e. [1],[2],[3] for red,green,blue).
def encode_text_index(df, name):
    le = preprocessing.LabelEncoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Encode a numeric column as zscores
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Convert all missing values in the specified column to the median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Convert all missing values in the specified column to the default
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Convert a Pandas dataframe to the x,y inputs that TensorFlow needs
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # find out the type of the target column.
    target_type = df[target].dtypes
    target_type = target_type[0] if isinstance(target_type, Sequence) else target_type
    # Encode to int for classification, float otherwise. TensorFlow likes 32 bits.
    #if target_type in (np.int64, np.int32):
        ## Classification
        #dummies = pd.get_dummies(df[target])
        #return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    #else#:
        ## Regression
    return df[result].values.astype(np.float32), df[target].values.astype(np.float32)

# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Regression chart.
def chart_regression(path, pred,y,sort=True):
    t = pd.DataFrame({'pred' : pred, 'y' : y.flatten()})
    if sort:
        t.sort_values(by=['y'],inplace=True)
    b = plt.plot(t['pred'].tolist(),label='prediction')
    a = plt.plot(t['y'].tolist(),label='expected')

    plt.ylabel('output')
    plt.legend()
    plt.savefig(path)
    plt.close()

# Remove all rows where the specified column is +/- sd standard deviations
def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Encode a column to a range between normalized_low and normalized_high.
def encode_numeric_range(df, name, normalized_low=-1, normalized_high=1,
                         data_low=None, data_high=None):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) \
               * (normalized_high - normalized_low) + normalized_low



In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rayonegautam/charanet")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'charanet' dataset.
Path to dataset files: /kaggle/input/charanet


## Configure Environment

In [14]:
import os

base_path = os.path.join(os.getcwd(), "output")
iteration_path = os.path.join(base_path, f"iteration-{ITERATION}")
mp3_dataset_base_path = os.path.join(path, "charaNet")
spectrogram_dataset_base_path = os.path.join(os.getcwd(), 'data/spectrogram_dataset')
spectrogram_train_path = os.path.join(spectrogram_dataset_base_path, 'train')
spectrogram_test_path = os.path.join(spectrogram_dataset_base_path, 'test')
spectrogram_val_path = os.path.join(spectrogram_dataset_base_path, 'val')
try:
    os.mkdir(os.path.join(os.getcwd(), "data"))
    os.mkdir(spectrogram_dataset_base_path)
    os.mkdir(spectrogram_train_path)
    os.mkdir(spectrogram_test_path)
    os.mkdir(spectrogram_val_path)
except FileExistsError as e:
    print('That Dataset Folder already exists')
    print(e)

try:
    os.mkdir(base_path)
except FileExistsError as e:
    print(f"{base_path} already exists")
except OSError as e:
    print(f"Error creating directory: {base_path}")
try:
    os.mkdir(iteration_path)
except FileExistsError as e:
    print(f"{iteration_path} already exists. Exiting to preserve previous work.")
    sys.exit(0)
except OSError:
    print("An error occurred while creating the folder. ")



## Import and Read Datasets

In [15]:
import librosa
import gc
def get_all_files(mp3_base_directory_path, spectrogram_base_directory_path, extension_filter = None):
    # i expect base directory path  to be one of:
    # charaNet/train charaNet/test charaNet/val.
    mp3_file_list = []
    spectrogram_file_list = []
    i = 0
    for species_directory in os.listdir(mp3_base_directory_path): # i expect these directories to map to the directories for individual species.
        # for each subdirectory
        mp3_speciesdirectory_path = os.path.join(mp3_base_directory_path, species_directory)
        spectrogram_speciesdirectory_path = os.path.join(spectrogram_base_directory_path, species_directory.replace(" ", "-"))
        try:
            print(f"making directory: {spectrogram_speciesdirectory_path}")
            os.mkdir(spectrogram_speciesdirectory_path)
        except OSError as e:
            print(f"an error occurred while making the spectrogram subdirectory path for directory: {species_directory.replace('', '-')}. \n{e}")
        for file_name in os.listdir(mp3_speciesdirectory_path):
          # for each file in the subdirectory
            print("mp3_speciesdirectory files:", file_name)

            if os.path.isfile(os.path.join(mp3_speciesdirectory_path, file_name)):
                if extension_filter is None:
                    mp3_file_list.append(os.path.join(mp3_speciesdirectory_path, file_name))
                    spectrogram_file_list.append(os.path.join(spectrogram_speciesdirectory_path, file_name.replace(" ", "")))
                else:
                    _, file_extension = os.path.splitext(file_name.replace(" ", ""))
                    if file_extension == extension_filter:
                        mp3_file_list.append(file_name)
                        spectrogram_file_list.append(os.path.join(spectrogram_speciesdirectory_path, file_name.replace(" ", "")))

    return (mp3_file_list, spectrogram_file_list)
def mp3_to_spectrogram(mp3_path, spectrogram_file_path):
    data, sample_rate = librosa.load(mp3_path)
    mel_spectrogram = librosa.feature.melspectrogram(y=data, sr=sample_rate)
    # Convert to Decibels (Log Scale)
    # Convert to decibels (log scale): Spectrograms are often displayed in decibels for better visualization of dynamic ranges.
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_spectrogram_db, x_axis='time', y_axis='mel', sr=sample_rate, cmap='viridis')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel Spectrogram')
    plt.savefig(spectrogram_file_path + ".png")
    plt.close('all')
    del data, mel_spectrogram, mel_spectrogram_db
    gc.collect()
train_mp3_base = os.path.join(mp3_dataset_base_path, 'train')
test_mp3_base = os.path.join(mp3_dataset_base_path, 'test')
val_mp3_base = os.path.join(mp3_dataset_base_path, 'val')
# grab all mp3 files in a directory
train_mp3_file_list, train_spectrogram_list = get_all_files(train_mp3_base, spectrogram_train_path)
test_mp3_file_list,test_spectro_list  = get_all_files(test_mp3_base, spectrogram_test_path)
val_mp3_file_list, val_spectro_list = get_all_files(val_mp3_base, spectrogram_val_path)
# is it safe to assume we can consider the lists linked? is it safe to simply use the same iterator for both lists?
for i , file in enumerate(train_mp3_file_list):
  if i % 100 == 0:
    print(f"processed {i} files")
    if i >= len(train_spectrogram_list):
        print("[FATAL] iterator exceeded spectrogram list length. consider some or many files to be lost! This function doesn't work!")
        break
    try:
      mp3_to_spectrogram(file, train_spectrogram_list[i])
    except ValueError as e:
      print("couldn't convert that file!", file)





Streaming output truncated to the last 5000 lines.
mp3_speciesdirectory files: XC310176 0.mp3
mp3_speciesdirectory files: XC413041 6.mp3
mp3_speciesdirectory files: XC703698 original.mp3
mp3_speciesdirectory files: XC650266 0.mp3
mp3_speciesdirectory files: XC413041 2.mp3
mp3_speciesdirectory files: XC406364 0.mp3
mp3_speciesdirectory files: XC720324 1.mp3
mp3_speciesdirectory files: XC699016 6.mp3
mp3_speciesdirectory files: XC720324 2.mp3
mp3_speciesdirectory files: XC487859 0.mp3
mp3_speciesdirectory files: XC310177 1.mp3
mp3_speciesdirectory files: XC611642 2.mp3
mp3_speciesdirectory files: XC721713 0.mp3
mp3_speciesdirectory files: XC699017 1.mp3
mp3_speciesdirectory files: XC522502 2.mp3
mp3_speciesdirectory files: XC420216 0.mp3
mp3_speciesdirectory files: XC501518 0.mp3
mp3_speciesdirectory files: XC698935 1.mp3
mp3_speciesdirectory files: XC487857 original.mp3
mp3_speciesdirectory files: XC487853 7.mp3
mp3_speciesdirectory files: XC699016 5.mp3
mp3_speciesdirectory files: XC69

Check to make sure all files were grabbed properly. If these numbers are short, something went wrong.

In [16]:
if DEBUG:

  print(f"lists:{len(train_mp3_file_list)}\n{len(test_mp3_file_list)}\n{len(val_mp3_file_list)}\n{train_mp3_file_list[:2]}\n{train_spectrogram_list[:2]}")


NameError: name 'DEBUG' is not defined

In [17]:
for i , file in enumerate(test_mp3_file_list):
  if i % 100 == 0:
    print(f"processed {i} files")
    if i >= len(test_spectro_list):
        print("[FATAL] iterator exceeded spectrogram list length. consider some or many files to be lost! This function doesn't work!")
        break
    try:
      mp3_to_spectrogram(file, test_spectro_list[i])
    except ValueError as e:
      print("couldn't convert that file!", file)

processed 0 files
processed 100 files
processed 200 files
processed 300 files
processed 400 files
processed 500 files
processed 600 files


In [18]:
for i , file in enumerate(val_mp3_file_list):
  if i % 100 == 0:
    print(f"processed {i} files")
    if i >= len(val_spectro_list):
        print("[FATAL] iterator exceeded spectrogram list length. consider some or many files to be lost! This function doesn't work!")
        break
    try:
      mp3_to_spectrogram(file, val_spectro_list[i])
    except ValueError as e:
      print("couldn't convert that file!", file)

processed 0 files
processed 100 files
processed 200 files
processed 300 files
processed 400 files
processed 500 files
processed 600 files
